In [4]:
from docplex.mp.model import Model
from datetime import timedelta

# --- CONSTANTS ---
HOURS = 24
MINUTES_IN_DAY = 1440
MinUtilTime = 400
MaxUtilTime = 1440

vehicle_fixed_cost = 1000
freq_penalty = 1000  # penalty for unmet frequency demand

nvehicle=20
K = range(nvehicle)  # vehicles
H = range(HOURS)  # hours

bigM = 1440  # big M for time constraints

# Frequencies of nodes per hour at ATB and HSK
freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB freq per hour
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK freq per hour

# Travel times and capacities for arcs (in minutes)
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {"ATB": "HSK", "HSK": "ATB"}
travel_time = {
    ("ATB", "HSK"): arc_data[("ATB", "HSK")][0],
    ("HSK", "ATB"): arc_data[("HSK", "ATB")][0]
}

# --- NODES CREATION ---
nodes = []
node_id = 1

# Start node
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
start_node_id = node_id
node_id += 1

# Create nodes per frequency for each hour at ATB and HSK
for hour in H:
    f1 = freq1[hour]
    f2 = freq2[hour]

    for i in range(f1):
        time_val = hour * 60 + (i * 60 // max(f1, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1

    for i in range(f2):
        time_val = hour * 60 + (i * 60 // max(f2, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

# End node
nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': 'X'})
end_node_id = node_id
node_id += 1

# Sort nodes by time
nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

# --- ARC CREATION ---
E = []
synthetic_id = node_id

# Arcs from X to ATB/HSK nodes reachable from start
for node in nodes:
    loc = node['loc']
    dst_time = node['time']

    if loc in ['ATB', 'HSK']:
        cost, N_bus = arc_data[('X', loc)]
        if dst_time >= cost:
            T = dst_time - cost
            V.append({'id': synthetic_id, 'time': T, 'loc': 'X'})
            E.append({
                'src': {'id': synthetic_id, 'time': T, 'loc': 'X'},
                'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
                'cost': cost,
                'N_bus': N_bus
            })
            synthetic_id += 1

# Chained arcs between ATB and HSK
for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time < MinUtilTime:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > MINUTES_IN_DAY:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        V.append(dst)
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'N_bus': 1
        })

        total_time += cost
        curr_loc = next_loc
        curr_time = next_time
        curr_src_id = dst['id']
        synthetic_id += 1

    cost_to_X, N_bus_to_X = arc_data[(curr_loc, 'X')]
    final_node = {'id': synthetic_id, 'time': curr_time + cost_to_X, 'loc': 'X'}
    V.append(final_node)
    E.append({
        'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
        'dst': final_node,
        'cost': cost_to_X,
        'N_bus': N_bus_to_X
    })
    synthetic_id += 1

# --- MODEL ---
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)

node_ids_in_arcs = set()
for e in E:
    node_ids_in_arcs.add(e['src']['id'])
    node_ids_in_arcs.add(e['dst']['id'])

arrival_time = mdl.continuous_var_dict(((k, nid) for k in K for nid in node_ids_in_arcs), name='arr', lb=0, ub=MINUTES_IN_DAY)

freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)

# Objective
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K)
    + mdl.sum(vehicle_fixed_cost * z[k] for k in K)
    + mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)
)

# Constraints
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E))

for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == start_node_id) == z[k])
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == end_node_id) == z[k])
    for node in V:
        i = node['id']
        if i != start_node_id and i != end_node_id:
            mdl.add_constraint(
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == i) ==
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == i)
            )

# Capacity (N_bus) == 1 arcs can be used only once across all vehicles
for e in E:
    if e['N_bus'] == 1:
        mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for k in K) <= 1)

for k in K:
    mdl.add_constraint(T[k] == mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E if e['N_bus'] == 1))
    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

for e in E:
    for k in K:
        i, j = e['src']['id'], e['dst']['id']
        cost = e['cost']
        mdl.add_constraint(arrival_time[k, j] >= arrival_time[k, i] + cost - bigM * (1 - x[i, j, k]))
        mdl.add_constraint(arrival_time[k, j] <= arrival_time[k, i] + cost + bigM * (1 - x[i, j, k]))

for h in H:
    mdl.add_constraint(
        mdl.sum(x[e['src']['id'], e['dst']['id'], k]
                for k in K
                for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h)
        + freq_slack_ATB[h] >= freq1[h]
    )
    mdl.add_constraint(
        mdl.sum(x[e['src']['id'], e['dst']['id'], k]
                for k in K
                for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h)
        + freq_slack_HSK[h] >= freq2[h]
    )

# --- SOLVE ---
solution = mdl.solve(log_output=True)

if solution:
    print("Objective:", solution.objective_value)
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")

        current_node = start_node_id

        while current_node != end_node_id:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break
            
            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']

            if src['loc'] == 'X':
                src_h, src_m = divmod(int(dst['time'] - arc['cost']), 60)
            else:
                src_h, src_m = divmod(src['time'], 60)

            if dst['loc'] == 'X':
                dst_h, dst_m = divmod(int(src['time'] + arc['cost']), 60)
            else:
                dst_h, dst_m = divmod(dst['time'], 60)

            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")


# Add readable time format
for node in V:
    node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

# Output all nodes in V
count = 0
for node in V:
    print(node)
    count += 1
print(f"Total nodes in V: {count}")

# Output arcs
print(f"Total generated arcs: {len(E)}")
for arc in E[:]:
    print(f"From node {arc['src']['id']} ({arc['src']['loc']} @ {arc['src']['time']}) "
          f"to node {arc['dst']['id']} ({arc['dst']['loc']} @ {arc['dst']['time']}) "
          f"cost: {arc['cost']}")


Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
CPXPARAM_LPMethod                                3
Found incumbent of value 42000.000000 after 0.00 sec. (0.71 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 11192 rows and 7648 columns.
MIP Presolve modified 3360 coefficients.
All rows and columns eliminated.
Presolve time = 0.02 sec. (6.71 ticks)

Root node processing (before b&c):
  Real time             =    0.02 sec. (7.68 ticks)
Parallel b&c, 32 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.02 sec. (7.68 ticks)
Objective: 42000.0
Number of vehicles used: 0

{'id': 1, 'time': 0, 'loc': 'X', 'hhmm': '0:00'}
{'id': 2, 'time': 1020, 'loc': 'ATB', 'hhmm': '17:00'}
{'id': 5, 'time': 1020, 'loc': 'HSK', 'hhmm': '17:00'}
{'id': 3, 'time': 1040, 'loc': 'ATB', 'hhmm':

In [5]:
from docplex.mp.model import Model
from datetime import timedelta

# --- CONSTANTS ---
HOURS = 24
MINUTES_IN_DAY = 1440
MinUtilTime = 400
MaxUtilTime = 1440

vehicle_fixed_cost = 1000
freq_penalty = 1000  # penalty for unmet frequency demand

nvehicle = 20
K = range(nvehicle)  # vehicles
H = range(HOURS)  # hours

bigM = 1440  # big M for time constraints

# Frequencies of nodes per hour at ATB and HSK
freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB freq per hour
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK freq per hour

# Travel times and capacities for arcs (in minutes)
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {"ATB": "HSK", "HSK": "ATB"}
travel_time = {
    ("ATB", "HSK"): arc_data[("ATB", "HSK")][0],
    ("HSK", "ATB"): arc_data[("HSK", "ATB")][0]
}

# --- NODES CREATION ---
nodes = []
node_id = 1

# Start node
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
start_node_id = node_id
node_id += 1

# Create nodes per frequency for each hour at ATB and HSK
for hour in H:
    f1 = freq1[hour]
    f2 = freq2[hour]

    for i in range(f1):
        time_val = hour * 60 + (i * 60 // max(f1, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1

    for i in range(f2):
        time_val = hour * 60 + (i * 60 // max(f2, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

# End node
nodes.append({'id': node_id, 'time': MINUTES_IN_DAY, 'loc': 'X'})
end_node_id = node_id
node_id += 1

# Sort nodes by time (not strictly necessary but okay)
nodes.sort(key=lambda x: x['time'])
V = nodes.copy()

# --- ARC CREATION ---
E = []
synthetic_id = node_id

# Add arcs from start node (start_node_id) to nodes at ATB or HSK reachable from X
for node in nodes:
    loc = node['loc']
    dst_time = node['time']
    if loc in ['ATB', 'HSK']:
        cost, N_bus = arc_data[('X', loc)]
        # Can only reach nodes after travel cost time from X (start)
        if dst_time >= cost:
            E.append({
                'src': {'id': synthetic_id, 'time': 0, 'loc': 'X'},
                'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
                'cost': cost,
                'N_bus': N_bus
            })
            synthetic_id += 1

# Chained arcs between ATB and HSK for each node where applicable
for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time < MinUtilTime:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > MINUTES_IN_DAY:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'N_bus': 1
        })

        total_time += cost
        curr_loc = next_loc
        curr_time = next_time
        curr_src_id = dst['id']
        synthetic_id += 1

    # Add arc to end node (X)
    cost_to_X, N_bus_to_X = arc_data[(curr_loc, 'X')]
    E.append({
        'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
        'dst': {'id': end_node_id, 'time': MINUTES_IN_DAY, 'loc': 'X'},
        'cost': cost_to_X,
        'N_bus': N_bus_to_X
    })

# --- MODEL ---
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
z = mdl.binary_var_dict(K, name='z')
T = mdl.continuous_var_dict(K, name='T', lb=0)

node_ids_in_arcs = set()
for e in E:
    node_ids_in_arcs.add(e['src']['id'])
    node_ids_in_arcs.add(e['dst']['id'])

arrival_time = mdl.continuous_var_dict(((k, nid) for k in K for nid in node_ids_in_arcs), name='arr', lb=0, ub=MINUTES_IN_DAY)

freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)

# Objective
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K)
    + mdl.sum(vehicle_fixed_cost * z[k] for k in K)
    + mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)
)

# Constraints
for k in K:
    # Link x and z (vehicle used)
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E))

for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == start_node_id) == z[k])
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == end_node_id) == z[k])
    for node in V:
        i = node['id']
        if i != start_node_id and i != end_node_id:
            mdl.add_constraint(
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == i) ==
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == i)
            )

# Capacity (N_bus == 1) arcs can be used only once across all vehicles
for e in E:
    if e['N_bus'] == 1:
        mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for k in K) <= 1)

for k in K:
    mdl.add_constraint(T[k] == mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E if e['N_bus'] == 1))
    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

for e in E:
    for k in K:
        i, j = e['src']['id'], e['dst']['id']
        cost = e['cost']
        mdl.add_constraint(arrival_time[k, j] >= arrival_time[k, i] + cost - bigM * (1 - x[i, j, k]))
        mdl.add_constraint(arrival_time[k, j] <= arrival_time[k, i] + cost + bigM * (1 - x[i, j, k]))

for h in H:
    mdl.add_constraint(
        mdl.sum(x[e['src']['id'], e['dst']['id'], k]
                for k in K
                for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h)
        + freq_slack_ATB[h] >= freq1[h]
    )
    mdl.add_constraint(
        mdl.sum(x[e['src']['id'], e['dst']['id'], k]
                for k in K
                for e in E if e['N_bus'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h)
        + freq_slack_HSK[h] >= freq2[h]
    )

# --- SOLVE ---
solution = mdl.solve(log_output=True)

if solution:
    print("Objective:", solution.objective_value)
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")

        current_node = start_node_id

        while current_node != end_node_id:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break

            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']

            # Format times nicely
            src_time_val = src['time']
            dst_time_val = dst['time']

            src_h, src_m = divmod(int(src_time_val), 60)
            dst_h, dst_m = divmod(int(dst_time_val), 60)

            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")


# Add readable time format to all nodes
for node in V:
    node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

# Output all nodes in V
count = 0
for node in V:
    print(node)
    count += 1
print(f"Total nodes in V: {count}")

# Output arcs summary
print(f"Total generated arcs: {len(E)}")
for arc in E:
    print(f"From node {arc['src']['id']} ({arc['src']['loc']} @ {arc['src']['time']}) "
          f"to node {arc['dst']['id']} ({arc['dst']['loc']} @ {arc['dst']['time']}) "
          f"cost: {arc['cost']}")


Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
CPXPARAM_LPMethod                                3
Found incumbent of value 42000.000000 after 0.00 sec. (0.59 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 7832 rows and 6828 columns.
MIP Presolve modified 6720 coefficients.
All rows and columns eliminated.
Presolve time = 0.00 sec. (5.81 ticks)

Root node processing (before b&c):
  Real time             =    0.02 sec. (6.63 ticks)
Parallel b&c, 32 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.02 sec. (6.63 ticks)
Objective: 42000.0
Number of vehicles used: 0

{'id': 1, 'time': 0, 'loc': 'X', 'hhmm': '0:00'}
{'id': 2, 'time': 1020, 'loc': 'ATB', 'hhmm': '17:00'}
{'id': 5, 'time': 1020, 'loc': 'HSK', 'hhmm': '17:00'}
{'id': 3, 'time': 1040, 'loc': 'ATB', 'hhmm': 